# v10 True Staged Batch API

This notebook runs the same step prompts/procedure as `notebooks/extraction_pipeline_v10.ipynb`, but executes each step via OpenAI Batch API in stages (Step1 -> Step2 -> Step3 -> Step4 -> Step5.1

In [ ]:
import json
import os
import time
from pathlib import Path

import pandas as pd
from openai import OpenAI

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("Set OPENAI_API_KEY in environment first.")

def first_existing(paths):
    for p in paths:
        if p.exists():
            return p
    return None

SOURCE_NB = first_existing([
    Path("notebooks/extraction_pipeline_v10.ipynb"),
    Path("../notebooks/extraction_pipeline_v10.ipynb"),
    Path("extraction_pipeline_v10.ipynb"),
    Path("../extraction_pipeline_v10.ipynb"),
])
if SOURCE_NB is None:
    raise FileNotFoundError(f"Could not locate extraction_pipeline_v10.ipynb from cwd={Path.cwd()}")

JSON_PATH = first_existing([
    Path("../data/raw/project-1-at-2026-04-20-12-53-8fd13aa5.json")
])
if JSON_PATH is None:
    raise FileNotFoundError(f"Could not locate input JSON from cwd={Path.cwd()}")

_output_base = first_existing([Path("data/output"), Path("../data/output")])
if _output_base is None:
    _output_base = Path("data/output")
OUTPUT_DIR = _output_base / "pipeline_gpt54_gpt52_210"
WORK_DIR = OUTPUT_DIR / "true_staged_batch_workdir"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)

# Step 1 model
MODEL_STEP1 = "gpt-5.4"
MODEL_COMPLEX = "gpt-5.2"
BATCH_POLL_SECONDS = 30

client = OpenAI(api_key=OPENAI_API_KEY)
MAX_ABSTRACTS = None  # set int (e.g., 20) to run a subset

print("Config ready")


In [ ]:
# Load step functions from v10 notebook (exact prompt/procedure source)
with open(SOURCE_NB, "r", encoding="utf-8") as f:
    src_nb = json.load(f)

exec_ns = {}
exec_ns["MODEL_SIMPLE"] = "gpt-4.1"
exec_ns["MODEL_COMPLEX"] = "gpt-5.2"

for cell in src_nb["cells"]:
    if cell.get("cell_type") != "code":
        continue
    src = "".join(cell.get("source", []))
    if "# Cell 12 — Batch execution" in src:
        break
    if "# Cell 2 — OpenAI client & model constants" in src:
        continue
    exec(src, exec_ns)

# Pull required symbols
clean_abstract = exec_ns["clean_abstract"]
parse_json_response = exec_ns["parse_json_response"]
step1_detect_variables = exec_ns["step1_detect_variables"]
step2_hierarchy_and_normalize = exec_ns["step2_hierarchy_and_normalize"]
step3_extract_relevant_sentences = exec_ns["step3_extract_relevant_sentences"]
step4_extract_relationships = exec_ns["step4_extract_relationships"]
step5_1_validate_relationships = exec_ns["step5_1_validate_relationships"]

def capture_prompt_blocks(step_fn, *args, **kwargs):
    holder = {}

    def _fake_complex(prompt_blocks, model=MODEL_COMPLEX):
        holder["prompt_blocks"] = prompt_blocks
        holder["call_type"] = "complex"
        holder["model"] = model
        return {"content": "{}", "usage": None}

    def _fake_simple(prompt_blocks, model=exec_ns.get("MODEL_SIMPLE", "gpt-4.1")):
        holder["prompt_blocks"] = prompt_blocks
        holder["call_type"] = "simple"
        holder["model"] = model
        return {"content": "{}", "usage": None}

    old_complex = exec_ns.get("call_openai_with_json_complex")
    old_simple = exec_ns.get("call_openai_with_json_simple")
    exec_ns["call_openai_with_json_complex"] = _fake_complex
    exec_ns["call_openai_with_json_simple"] = _fake_simple
    try:
        step_fn(*args, **kwargs)
    finally:
        exec_ns["call_openai_with_json_complex"] = old_complex
        exec_ns["call_openai_with_json_simple"] = old_simple

    return holder

print("Loaded prompt builders from source notebook")

In [ ]:
def build_batch_request(custom_id, captured):
    prompt_blocks = captured["prompt_blocks"]
    call_type = captured.get("call_type", "complex")
    model = captured.get("model", MODEL_COMPLEX)

    if custom_id.startswith("step1::") or custom_id.startswith("step5_1::"):
        model = MODEL_STEP1

    body = {
        "model": model,
        "input": prompt_blocks,
        "text": {"format": {"type": "text"}},
        "tools": [],
        "store": True,
    }

    if call_type == "complex" or custom_id.startswith("step1::") or custom_id.startswith("step5_1::"):
        body["reasoning"] = {"effort": "low", "summary": "auto"}

    return {
        "custom_id": custom_id,
        "method": "POST",
        "url": "/v1/responses",
        "body": body,
    }


def extract_output_text(body):
    texts = []
    for item in body.get("output", []):
        if item.get("type") == "message":
            for c in item.get("content", []):
                if c.get("type") == "output_text" and c.get("text"):
                    texts.append(c["text"])
    return "\n".join(texts).strip()


def run_batch_stage(stage_name, requests):
    stage_dir = WORK_DIR / stage_name
    stage_dir.mkdir(parents=True, exist_ok=True)

    input_jsonl = stage_dir / "input.jsonl"
    with open(input_jsonl, "w", encoding="utf-8") as f:
        for r in requests:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

    with open(input_jsonl, "rb") as f:
        in_file = client.files.create(file=f, purpose="batch")

    batch = client.batches.create(
        input_file_id=in_file.id,
        endpoint="/v1/responses",
        completion_window="24h",
    )

    print(f"[{stage_name}] batch_id={batch.id}, requests={len(requests)}")

    while True:
        b = client.batches.retrieve(batch.id)
        print(f"[{stage_name}] status={b.status} counts={getattr(b, 'request_counts', None)}")
        if b.status in {"completed", "failed", "expired", "cancelled"}:
            break
        time.sleep(BATCH_POLL_SECONDS)

    req_counts = getattr(b, "request_counts", None)
    failed_n = int(getattr(req_counts, "failed", 0) or 0) if req_counts is not None else 0

    if getattr(b, "error_file_id", None):
        err_bytes = client.files.content(b.error_file_id).read()
        with open(stage_dir / "errors.jsonl", "wb") as f:
            f.write(err_bytes)

    if b.status != "completed":
        raise RuntimeError(f"Stage {stage_name} failed with status={b.status}")

    if failed_n > 0:
        print(f"[{stage_name}] WARNING: {failed_n} requests failed at API layer")

    if not getattr(b, "output_file_id", None):
        err_hint = stage_dir / "errors.jsonl"
        raise RuntimeError(
            f"Stage {stage_name} completed without output_file_id; failed={failed_n}. "
            f"See {err_hint} for details."
        )

    out_bytes = client.files.content(b.output_file_id).read()
    out_jsonl = stage_dir / "output.jsonl"
    with open(out_jsonl, "wb") as f:
        f.write(out_bytes)

    records = {}
    with open(out_jsonl, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            rec = json.loads(line)
            cid = rec.get("custom_id", "")
            response_obj = rec.get("response", {})
            status_code = int(response_obj.get("status_code", 200) or 200)
            body = response_obj.get("body", {})
            if status_code != 200:
                records[cid] = {
                    "ok": False,
                    "error": f"HTTP {status_code}: {json.dumps(body, ensure_ascii=False)[:2000]}",
                    "raw": "",
                    "tokens": {"input_tokens": 0, "output_tokens": 0, "total_tokens": 0},
                }
                continue

            text = extract_output_text(body)
            usage = body.get("usage", {}) or {}
            try:
                if not text:
                    raise ValueError("empty output_text")
                parsed = parse_json_response({"content": text})
                records[cid] = {
                    "ok": True,
                    "result": parsed,
                    "raw": text,
                    "tokens": {
                        "input_tokens": int(usage.get("input_tokens", 0) or 0),
                        "output_tokens": int(usage.get("output_tokens", 0) or 0),
                        "total_tokens": int(usage.get("total_tokens", 0) or 0),
                    },
                }
            except Exception as e:
                records[cid] = {"ok": False, "error": str(e), "raw": text, "tokens": {"input_tokens": 0, "output_tokens": 0, "total_tokens": 0}}


    if failed_n > 0:
        bad = [k for k, v in records.items() if not v.get("ok", False)]
        print(f"[{stage_name}] Parsed failed records: {len(bad)}")
        if bad:
            sample = bad[0]
            print(f"[{stage_name}] Sample failure {sample}: {records[sample].get('error','')}")

    return records


def _role_strs(lst):
    out = []
    for x in lst:
        if isinstance(x, dict):
            name = x.get("variable", "")
            comps = x.get("components", [])
            out.append(f"{name} [{', '.join(comps)}]" if comps else name)
        else:
            out.append(str(x))
    return out

print(f"Batch helpers ready | step1 model={MODEL_STEP1} | other complex model={MODEL_COMPLEX}")


In [ ]:
# Load articles
with open(JSON_PATH, "r", encoding="utf-8") as f:
    articles = json.load(f)

items = []
for a in articles:
    aid = int(a["id"])
    key = f"json_{aid}"
    raw_abs = a.get("data", {}).get("AB", "")
    abstract = clean_abstract(raw_abs) if raw_abs else ""
    if not abstract:
        continue
    items.append({
        "article_id": key,
        "title": a.get("data", {}).get("TI", ""),
        "abstract": abstract,
    })

MAX_ABSTRACTS = None

if MAX_ABSTRACTS is not None:
    items = items[: int(MAX_ABSTRACTS)]

by_id = {x["article_id"]: x for x in items}
print(f"Loaded {len(items)} articles with abstracts (MAX_ABSTRACTS={MAX_ABSTRACTS})")

# ---------- Stage 1 ----------
reqs = []
for x in items:
    pb = capture_prompt_blocks(step1_detect_variables, x["abstract"])
    reqs.append(build_batch_request(f"step1::{x['article_id']}", pb))
stage1 = run_batch_stage("step1", reqs)

# ---------- Stage 2 ----------
reqs = []
for x in items:
    s1 = stage1.get(f"step1::{x['article_id']}", {}).get("result", {})
    pb = capture_prompt_blocks(step2_hierarchy_and_normalize, x["abstract"], s1)
    reqs.append(build_batch_request(f"step2::{x['article_id']}", pb))
stage2 = run_batch_stage("step2", reqs)

# ---------- Stage 3 ----------
reqs = []
for x in items:
    s1 = stage1.get(f"step1::{x['article_id']}", {}).get("result", {})
    s2 = stage2.get(f"step2::{x['article_id']}", {}).get("result", {})
    final_vars = s2.get("final_variable_list", s1.get("final_variable_list", []))
    pb = capture_prompt_blocks(step3_extract_relevant_sentences, x["abstract"], final_vars)
    reqs.append(build_batch_request(f"step3::{x['article_id']}", pb))
stage3 = run_batch_stage("step3", reqs)

# ---------- Stage 4 ----------
reqs = []
for x in items:
    s1 = stage1.get(f"step1::{x['article_id']}", {}).get("result", {})
    s2 = stage2.get(f"step2::{x['article_id']}", {}).get("result", {})
    s3 = stage3.get(f"step3::{x['article_id']}", {}).get("result", {})
    final_vars = s2.get("final_variable_list", s1.get("final_variable_list", []))
    sentences = s3.get("relevant_sentences", [])
    pb = capture_prompt_blocks(step4_extract_relationships, x["abstract"], final_vars, sentences)
    reqs.append(build_batch_request(f"step4::{x['article_id']}", pb))
stage4 = run_batch_stage("step4", reqs)

# Build Step1-4 result table (same structure as source Cell 12)
rows = []
for x in items:
    aid = x["article_id"]
    s1r = stage1.get(f"step1::{aid}", {"result": {}, "tokens": {"input_tokens":0,"output_tokens":0,"total_tokens":0}})
    s2r = stage2.get(f"step2::{aid}", {"result": {}, "tokens": {"input_tokens":0,"output_tokens":0,"total_tokens":0}})
    s3r = stage3.get(f"step3::{aid}", {"result": {}, "tokens": {"input_tokens":0,"output_tokens":0,"total_tokens":0}})
    s4r = stage4.get(f"step4::{aid}", {"result": {}, "tokens": {"input_tokens":0,"output_tokens":0,"total_tokens":0}})

    s1, s2, s3, s4 = s1r.get("result", {}), s2r.get("result", {}), s3r.get("result", {}), s4r.get("result", {})
    s1_vars = s1.get("final_variable_list", [])
    final_vars = s2.get("final_variable_list", s1_vars)
    hier_rels = s2.get("hierarchy", {}).get("hierarchical_relationships", [])
    hier_strs = [f"{h.get('higher_level_variable','')} -> {h.get('lower_level_variable','')}" for h in hier_rels if isinstance(h, dict)]
    sentences = s3.get("relevant_sentences", [])
    dir_strs = [f"{r[0]} -> {r[2]} [{r[3]}]" for r in s4.get("directional", []) if len(r) >= 4]
    cor_strs = [f"{r[0]} <-> {r[2]} [{r[3]}]" for r in s4.get("correlational", []) if len(r) >= 4]
    mod_strs = [f"{m.get('moderator','?')} moderates ({', '.join(m.get('moderated_variables',[]))}) [{m.get('validation','?')}]" for m in s4.get("moderation", [])]

    t_in = sum(z.get("tokens", {}).get("input_tokens", 0) for z in [s1r, s2r, s3r, s4r])
    t_out = sum(z.get("tokens", {}).get("output_tokens", 0) for z in [s1r, s2r, s3r, s4r])

    rows.append({
        "article_id": aid,
        "title": x["title"],
        "abstract": x["abstract"],
        "step1_variables": "; ".join(s1_vars),
        "step1_count": len(s1_vars),
        "step1_IVs": "; ".join(s1.get("independent_variables", [])) if s1.get("independent_variables") else "",
        "step1_DVs": "; ".join(s1.get("dependent_variables", [])) if s1.get("dependent_variables") else "",
        "step1_moderators": "; ".join(s1.get("moderators", [])),
        "step1_mediators": "; ".join(s1.get("mediators", [])),
        "step2_final_vars": "; ".join(final_vars),
        "step2_final_count": len(final_vars),
        "step2_IVs": "; ".join(_role_strs(s2.get("independent_variables", []))),
        "step2_DVs": "; ".join(_role_strs(s2.get("dependent_variables", []))),
        "step2_moderators": "; ".join(s2.get("moderators", [])) if s2.get("moderators") else "",
        "step2_mediators": "; ".join(s2.get("mediators", [])) if s2.get("mediators") else "",
        "step2_other_controls": "; ".join([f"{o.get('name','')} ({o.get('role','')})" if isinstance(o, dict) else str(o) for o in s2.get("other_variables_controls", [])]),
        "step2_hierarchy": "; ".join(hier_strs),
        "step2_hierarchy_count": len(hier_rels),
        "step3_sentences": "; ".join(sentences),
        "step3_sentence_count": len(sentences),
        "step4_directional": "; ".join(dir_strs),
        "step4_directional_count": len(s4.get("directional", [])),
        "step4_correlational": "; ".join(cor_strs),
        "step4_correlational_count": len(s4.get("correlational", [])),
        "step4_moderation": "; ".join(mod_strs),
        "step4_moderation_count": len(s4.get("moderation", [])),
        "input_tokens": t_in,
        "output_tokens": t_out,
        "total_tokens": t_in + t_out,
    })

df_results = pd.DataFrame(rows)
csv_steps = OUTPUT_DIR / "pipeline_v4_results_330version.csv"
df_results.to_csv(csv_steps, index=False)
print(f"Saved steps 1-4: {csv_steps}")

# ---------- Stage 5.1 ----------
print("\n=== Running Stage 5.1 ===")
reqs = []
for _, r in df_results.iterrows():
    abstract = str(r.get("abstract", ""))  # Use full abstract from CSV
    pb = capture_prompt_blocks(
        step5_1_validate_relationships,
        abstract=abstract,
        step2_hierarchy=str(r.get("step2_hierarchy", "")),
        step4_directional=str(r.get("step4_directional", "")),
        step4_correlational=str(r.get("step4_correlational", "")),
        step4_moderation=str(r.get("step4_moderation", "")),
        variable_list=str(r.get("step2_final_vars", "")),
    )
    reqs.append(build_batch_request(f"step5_1::{r['article_id']}", pb))
stage5_1 = run_batch_stage("step5_1", reqs)

# Build validated table (add step5_1 results to existing rows)
validated_rows = []
for _, row in df_results.iterrows():
    aid = row["article_id"]
    s51r = stage5_1.get(f"step5_1::{aid}", {"result": {}, "tokens": {"input_tokens":0,"output_tokens":0,"total_tokens":0}})
    s51 = s51r.get("result", {})

    hier_5_1_strs = [f"{h[0]} -> {h[2]}" for h in s51.get("hierarchy", []) if isinstance(h, list) and len(h) >= 3]
    dir_5_1_strs = [f"{r[0]} -> {r[2]} [{r[3]}]" for r in s51.get("directional", []) if isinstance(r, list) and len(r) >= 4]
    cor_5_1_strs = [f"{r[0]} <-> {r[2]} [{r[3]}]" for r in s51.get("correlational", []) if isinstance(r, list) and len(r) >= 4]
    mod_5_1_strs = [f"{m.get('moderator','?')} moderates ({', '.join(m.get('moderated_variables',[]))}) [{m.get('validation','?')}]" for m in s51.get("moderation", [])]
    just_5_1_strs = [f"{j.get('change','')}: {j.get('reason','')}" for j in s51.get("justifications", [])]

    new_row = row.to_dict()
    new_row.update({
        "step5_1_hierarchy": "; ".join(hier_5_1_strs),
        "step5_1_directional": "; ".join(dir_5_1_strs),
        "step5_1_correlational": "; ".join(cor_5_1_strs),
        "step5_1_moderation": "; ".join(mod_5_1_strs),
        "step5_1_justifications": " | ".join(just_5_1_strs),
        "step5_1_input_tokens": s51r.get("tokens", {}).get("input_tokens", 0),
        "step5_1_output_tokens": s51r.get("tokens", {}).get("output_tokens", 0),
        "step5_1_total_tokens": s51r.get("tokens", {}).get("total_tokens", 0),
    })
    validated_rows.append(new_row)

df_validated = pd.DataFrame(validated_rows)
csv_validated = OUTPUT_DIR / "pipeline_v4_results_330version_validated.csv"
df_validated.to_csv(csv_validated, index=False)

print(f"\nSaved validated (with step5.1): {csv_validated}")
print("Done true staged batch pipeline (Steps 1-4 + 5.1).")


In [ ]:
# ========== RERUN ONLY FAILED ARTICLES ==========
# Extract failed article IDs from error file and rerun only those articles

ERROR_FILE = OUTPUT_DIR / "batch_69e75613a80c8190a65e2411d6823f1c_error.jsonl"

if ERROR_FILE.exists():
    print(f"Found error file: {ERROR_FILE}")
    
    # Extract failed article IDs
    failed_article_ids = set()
    with open(ERROR_FILE, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                err_rec = json.loads(line)
                custom_id = err_rec.get("custom_id", "")
                if custom_id and "::" in custom_id:
                    # Extract article_id from custom_id like "step1::json_25"
                    article_id = custom_id.split("::")[1]
                    failed_article_ids.add(article_id)
            except json.JSONDecodeError:
                continue
    
    print(f"Found {len(failed_article_ids)} failed articles: {sorted(failed_article_ids)}")
    
    if failed_article_ids:
        # Load only the failed articles from JSON
        with open(JSON_PATH, "r", encoding="utf-8") as f:
            all_articles = json.load(f)
        
        failed_items = []
        for a in all_articles:
            aid = int(a["id"])
            key = f"json_{aid}"
            if key in failed_article_ids:
                raw_abs = a.get("data", {}).get("AB", "")
                abstract = clean_abstract(raw_abs) if raw_abs else ""
                if abstract:
                    failed_items.append({
                        "article_id": key,
                        "title": a.get("data", {}).get("TI", ""),
                        "abstract": abstract,
                    })
        
        print(f"Loaded {len(failed_items)} failed articles from JSON")
        
        # Run complete pipeline for failed articles only
        # ---------- Retry Stage 1 ----------
        print("\n=== Retry Stage 1 ===")
        reqs = []
        for x in failed_items:
            pb = capture_prompt_blocks(step1_detect_variables, x["abstract"])
            reqs.append(build_batch_request(f"step1::{x['article_id']}", pb))
        retry_stage1 = run_batch_stage("step1_failed_retry", reqs)
        
        # ---------- Retry Stage 2 ----------
        print("\n=== Retry Stage 2 ===")
        reqs = []
        for x in failed_items:
            s1 = retry_stage1.get(f"step1::{x['article_id']}", {}).get("result", {})
            pb = capture_prompt_blocks(step2_hierarchy_and_normalize, x["abstract"], s1)
            reqs.append(build_batch_request(f"step2::{x['article_id']}", pb))
        retry_stage2 = run_batch_stage("step2_failed_retry", reqs)
        
        # ---------- Retry Stage 3 ----------
        print("\n=== Retry Stage 3 ===")
        reqs = []
        for x in failed_items:
            s1 = retry_stage1.get(f"step1::{x['article_id']}", {}).get("result", {})
            s2 = retry_stage2.get(f"step2::{x['article_id']}", {}).get("result", {})
            final_vars = s2.get("final_variable_list", s1.get("final_variable_list", []))
            pb = capture_prompt_blocks(step3_extract_relevant_sentences, x["abstract"], final_vars)
            reqs.append(build_batch_request(f"step3::{x['article_id']}", pb))
        retry_stage3 = run_batch_stage("step3_failed_retry", reqs)
        
        # ---------- Retry Stage 4 ----------
        print("\n=== Retry Stage 4 ===")
        reqs = []
        for x in failed_items:
            s1 = retry_stage1.get(f"step1::{x['article_id']}", {}).get("result", {})
            s2 = retry_stage2.get(f"step2::{x['article_id']}", {}).get("result", {})
            s3 = retry_stage3.get(f"step3::{x['article_id']}", {}).get("result", {})
            final_vars = s2.get("final_variable_list", s1.get("final_variable_list", []))
            sentences = s3.get("relevant_sentences", [])
            pb = capture_prompt_blocks(step4_extract_relationships, x["abstract"], final_vars, sentences)
            reqs.append(build_batch_request(f"step4::{x['article_id']}", pb))
        retry_stage4 = run_batch_stage("step4_failed_retry", reqs)
        
        # Build retry results table
        retry_rows = []
        for x in failed_items:
            aid = x["article_id"]
            s1r = retry_stage1.get(f"step1::{aid}", {"result": {}, "tokens": {"input_tokens":0,"output_tokens":0,"total_tokens":0}})
            s2r = retry_stage2.get(f"step2::{aid}", {"result": {}, "tokens": {"input_tokens":0,"output_tokens":0,"total_tokens":0}})
            s3r = retry_stage3.get(f"step3::{aid}", {"result": {}, "tokens": {"input_tokens":0,"output_tokens":0,"total_tokens":0}})
            s4r = retry_stage4.get(f"step4::{aid}", {"result": {}, "tokens": {"input_tokens":0,"output_tokens":0,"total_tokens":0}})

            s1, s2, s3, s4 = s1r.get("result", {}), s2r.get("result", {}), s3r.get("result", {}), s4r.get("result", {})
            s1_vars = s1.get("final_variable_list", [])
            final_vars = s2.get("final_variable_list", s1_vars)
            hier_rels = s2.get("hierarchy", {}).get("hierarchical_relationships", [])
            hier_strs = [f"{h.get('higher_level_variable','')} -> {h.get('lower_level_variable','')}" for h in hier_rels if isinstance(h, dict)]
            sentences = s3.get("relevant_sentences", [])
            dir_strs = [f"{r[0]} -> {r[2]} [{r[3]}]" for r in s4.get("directional", []) if len(r) >= 4]
            cor_strs = [f"{r[0]} <-> {r[2]} [{r[3]}]" for r in s4.get("correlational", []) if len(r) >= 4]
            mod_strs = [f"{m.get('moderator','?')} moderates ({', '.join(m.get('moderated_variables',[]))}) [{m.get('validation','?')}]" for m in s4.get("moderation", [])]

            t_in = sum(z.get("tokens", {}).get("input_tokens", 0) for z in [s1r, s2r, s3r, s4r])
            t_out = sum(z.get("tokens", {}).get("output_tokens", 0) for z in [s1r, s2r, s3r, s4r])

            retry_rows.append({
                "article_id": aid,
                "title": x["title"],
                "abstract": x["abstract"],
                "step1_variables": "; ".join(s1_vars),
                "step1_count": len(s1_vars),
                "step1_IVs": "; ".join(s1.get("independent_variables", [])) if s1.get("independent_variables") else "",
                "step1_DVs": "; ".join(s1.get("dependent_variables", [])) if s1.get("dependent_variables") else "",
                "step1_moderators": "; ".join(s1.get("moderators", [])),
                "step1_mediators": "; ".join(s1.get("mediators", [])),
                "step2_final_vars": "; ".join(final_vars),
                "step2_final_count": len(final_vars),
                "step2_IVs": "; ".join(_role_strs(s2.get("independent_variables", []))),
                "step2_DVs": "; ".join(_role_strs(s2.get("dependent_variables", []))),
                "step2_moderators": "; ".join(s2.get("moderators", [])) if s2.get("moderators") else "",
                "step2_mediators": "; ".join(s2.get("mediators", [])) if s2.get("mediators") else "",
                "step2_other_controls": "; ".join([f"{o.get('name','')} ({o.get('role','')})" if isinstance(o, dict) else str(o) for o in s2.get("other_variables_controls", [])]),
                "step2_hierarchy": "; ".join(hier_strs),
                "step2_hierarchy_count": len(hier_rels),
                "step3_sentences": "; ".join(sentences),
                "step3_sentence_count": len(sentences),
                "step4_directional": "; ".join(dir_strs),
                "step4_directional_count": len(s4.get("directional", [])),
                "step4_correlational": "; ".join(cor_strs),
                "step4_correlational_count": len(s4.get("correlational", [])),
                "step4_moderation": "; ".join(mod_strs),
                "step4_moderation_count": len(s4.get("moderation", [])),
                "input_tokens": t_in,
                "output_tokens": t_out,
                "total_tokens": t_in + t_out,
            })

        df_retry_results = pd.DataFrame(retry_rows)
        csv_retry = OUTPUT_DIR / "pipeline_v4_results_failed_articles_retry.csv"
        df_retry_results.to_csv(csv_retry, index=False)
        print(f"\nSaved retry steps 1-4: {csv_retry}")
        
        # ---------- Retry Stage 5.1 ----------
        print("\n=== Retry Stage 5.1 ===")
        reqs = []
        for _, r in df_retry_results.iterrows():
            abstract = str(r.get("abstract", ""))
            pb = capture_prompt_blocks(
                step5_1_validate_relationships,
                abstract=abstract,
                step2_hierarchy=str(r.get("step2_hierarchy", "")),
                step4_directional=str(r.get("step4_directional", "")),
                step4_correlational=str(r.get("step4_correlational", "")),
                step4_moderation=str(r.get("step4_moderation", "")),
                variable_list=str(r.get("step2_final_vars", "")),
            )
            reqs.append(build_batch_request(f"step5_1::{r['article_id']}", pb))
        retry_stage5_1 = run_batch_stage("step5_1_failed_retry", reqs)
        
        # Build validated retry table
        retry_validated_rows = []
        for _, row in df_retry_results.iterrows():
            aid = row["article_id"]
            s51r = retry_stage5_1.get(f"step5_1::{aid}", {"result": {}, "tokens": {"input_tokens":0,"output_tokens":0,"total_tokens":0}})
            s51 = s51r.get("result", {})

            hier_5_1_strs = [f"{h[0]} -> {h[2]}" for h in s51.get("hierarchy", []) if isinstance(h, list) and len(h) >= 3]
            dir_5_1_strs = [f"{r[0]} -> {r[2]} [{r[3]}]" for r in s51.get("directional", []) if isinstance(r, list) and len(r) >= 4]
            cor_5_1_strs = [f"{r[0]} <-> {r[2]} [{r[3]}]" for r in s51.get("correlational", []) if isinstance(r, list) and len(r) >= 4]
            mod_5_1_strs = [f"{m.get('moderator','?')} moderates ({', '.join(m.get('moderated_variables',[]))}) [{m.get('validation','?')}]" for m in s51.get("moderation", [])]
            just_5_1_strs = [f"{j.get('change','')}: {j.get('reason','')}" for j in s51.get("justifications", [])]

            new_row = row.to_dict()
            new_row.update({
                "step5_1_hierarchy": "; ".join(hier_5_1_strs),
                "step5_1_directional": "; ".join(dir_5_1_strs),
                "step5_1_correlational": "; ".join(cor_5_1_strs),
                "step5_1_moderation": "; ".join(mod_5_1_strs),
                "step5_1_justifications": " | ".join(just_5_1_strs),
                "step5_1_input_tokens": s51r.get("tokens", {}).get("input_tokens", 0),
                "step5_1_output_tokens": s51r.get("tokens", {}).get("output_tokens", 0),
                "step5_1_total_tokens": s51r.get("tokens", {}).get("total_tokens", 0),
            })
            retry_validated_rows.append(new_row)

        df_retry_validated = pd.DataFrame(retry_validated_rows)
        csv_retry_validated = OUTPUT_DIR / "pipeline_v4_results_failed_articles_retry_validated.csv"
        df_retry_validated.to_csv(csv_retry_validated, index=False)
        
        print(f"Saved retry validated (with step5.1): {csv_retry_validated}")
        print(f"\n✓ Done! Reran {len(failed_items)} failed articles through complete pipeline (steps 1-5.1)")
        
else:
    print(f"No error file found at {ERROR_FILE}")
